## 1. Introduction

**Unmanned Aerial Vehicles (UAVs)** have rapidly evolved from simple remote-controlled platforms into highly autonomous, complex **Cyber-Physical Systems (CPS)**. As their deployment expands across critical infrastructure, defense logistics, and autonomous surveillance, their attack surface has scaled proportionally. Securing these systems presents a unique and formidable challenge: a UAV is simultaneously a flying physical asset governed by aerodynamic kinematics and a networked computer reliant on continuous, secure data transmission. Because they inherently bridge the digital and physical worlds, a vulnerability in one domain can quickly cascade into failure in the other. Consequently, ensuring the operational integrity and security of UAV fleets is no longer just an aerospace engineering challenge, but a paramount cybersecurity imperative.

Despite the critical nature of these systems, traditional security paradigms face inherent limitations when protecting against adversaries. Contemporary **Intrusion Detection Systems (IDS)** and aerospace condition-monitoring frameworks frequently operate in isolated silos, separating network packet analysis from physical kinematic telemetry. Furthermore, these conventional defenses rely heavily on predefined threat signatures—such as known malware hashes or static physical thresholds. While effective against documented threats, this creates a vulnerability: zero-day cyber-physical attacks, such as **Command & Control (C&C) hijacking** or **latent denial-of-service packet floods**, routinely evade detection. Instead of triggering hard thresholds, these attacks often manifest as subtle, sustained statistical deviations across multiple sensors simultaneously. When existing monitoring systems attempt to flag these complex interactions without an understanding of historical temporal context, they generate overwhelming volumes of false positives. This induces severe **"alert fatigue"**, overwhelming **Security Operations Centers (SOCs)** and resulting in genuine, multi-vector threats being buried under systemic noise.

To address these ongoing challenges, this paper presents an integrated architecture predicated on **Dual-Sensor Fusion** and **temporal meta-ensembles**. This framework is driven by three foundational hypotheses: first, **the Kinematic Hypothesis**, which posits that physical hardware manipulation produces distinct mathematical signatures in physical telemetry prior to critical failure; second, **the Cyber Hypothesis**, asserting that latent network threats manifest as statistical outliers in network metadata without requiring known malicious payloads; and third, **the Temporal State Hypothesis**, which suggests that cyber-physical anomalies are sustained events best isolated using historical memory rather than point-in-time analysis. To operationalize these hypotheses, we implemented a unified pipeline expanding raw telemetry into a **195-dimensional** temporal feature space. Within this high-dimensional space, we evaluated a comprehensive matrix of **six unsupervised experiments**: a **tree-based Isolation Forest**, a **boundary-driven One-Class SVM**, a **density-based Local Outlier Factor (LOF)** paired with **PCA**, a **K-Means geometric hybrid**, and **two relational Meta-Ensembles**. By comparatively measuring the unsupervised decision boundary gap and resistance to dimensional sparsity, the **Isolation Forest** was selected as the primary modeling engine and further maximized via **Optuna Bayesian hyperparameter tuning**. To ensure operational transparency, we conducted a **deep error analysis** on this model, interrogating its borderline classifications to identify inherent architectural limitations, such as **density masking** and the **temporal smearing of micro-burst attacks**. Crucially, to mitigate the overarching issue of alert fatigue and offset these individual model vulnerabilities, this research implements **Majority-Vote Meta-Ensemble Tribunals**. By requiring consensus across diverse algorithms and domains, this framework effectively distills hundreds of noisy, false-positive alerts into a highly refined list of actionable zero-day threats.

## 2. Related work

The challenge of securing UAVs has historically been approached from two isolated domains: conventional **Information Technology (IT) network security** and **Operational Technology (OT) aerospace condition monitoring**. Contemporary IDS focus heavily on deep learning techniques applied to network packet analysis to identify malicious payloads [2]. While effective against documented cyber threats, these network-centric systems are fundamentally blind to the physical consequences of an attack on a UAV's kinematics. Conversely, aerospace condition monitoring increasingly relies on data-driven approaches to detect spatiotemporal anomalies in physical flight data [11], yet these systems often lack the network context required to differentiate a genuine mechanical failure from a sophisticated cyber-attack. Recent research has attempted to bridge this gap by exploring AI-based anomaly detection within **UAV Ground Control Systems (GCS)** [8], and the emergence of modern, high-dimensional drone communication datasets emphasizes the necessity of analyzing cyber and physical domains simultaneously [15, 16]. However, effectively fusing these domains without triggering systemic false alarms remains a persistent challenge in the literature.

To detect zero-day cyber-physical threats that evade signature-based systems, researchers have increasingly turned to unsupervised machine learning. Automated machine learning pipelines evaluating models such as **LOF**, **OC-SVM**, and **Isolation Forests** have demonstrated strong theoretical efficacy when applied to UAV telemetry [10]. These modern applications build upon foundational anomaly detection mathematics—such as **density-based local outlier identification** [3], **recursive random partitioning** [4], and **high-dimensional support estimation** [9] — adapting them to the complexities of continuous, unsupervised flight data [6]. Despite these algorithmic advancements, a significant limitation persists regarding the **"curse of dimensionality"**. As UAV telemetry expands to concurrently monitor both network packets and physical kinematics, distance-based anomaly detection models suffer from dimensional sparsity. In these highly dimensional spaces, the boundary between normal operations and anomalous events frequently collapses, severely degrading the model's precision.

This dimensional sparsity directly contributes to the most critical operational bottleneck in modern cybersecurity: alert fatigue. Many proposed unsupervised models evaluate telemetry strictly as **point-in-time anomalies**. Without transparent, multi-layer log analysis [5] or historical temporal context, these models flag brief, benign sensor noise as critical threats. In a live SOC, this generates hundreds of unactionable alerts, leading human operators to eventually dismiss genuine threats. To counteract this noise, recent cybersecurity research has proposed **adaptive ensemble learning-based IDS architectures** to enhance detection robustness in networked environments [7].

Building upon these findings, our research explicitly addresses alert fatigue by coupling temporal feature engineering with **Majority-Vote Meta-Ensemble Tribunals**. Furthermore, while next-generation hyperparameter optimization frameworks like **Optuna** [1] are frequently utilized to maximize predictive accuracy in supervised tasks, our work uniquely adapts these tools to maximize the unsupervised decision boundary gap, ensuring the resulting ensemble is primed to isolate sustained, multi-vector zero-day attacks.

## 3. Data

The foundation of this research relies on high-fidelity telemetry spanning both the physical and network operational domains of UAVs. To establish a baseline for the physical kinematics and operational behavior of autonomous fleets, this study leverages the **SurveilDrone-Net23 dataset** [15]. Compiled under the supervision of the Royal Institute of Drone Technologies (RIDT) in Oslo, this corpus captures over **140,256 continuous telemetry logs** collected at **15-minute intervals** between **January 2021 and December 2024**. The dataset provides an exhaustive multi-dimensional view of UAV flight mechanics, capturing **3D kinematic variables** (e.g., spatial velocity, tri-axial acceleration, altitude, and heading) alongside essential power consumption metrics and environmental proximity conditions. Originally designed to support synthetic-to-real UAV pattern generalization and surveillance behavior classification, this dataset provides the mechanical baseline necessary to test our **Kinematic Hypothesis**. By establishing the normal aerodynamic and power-drain parameters of routine behaviors—such as patrolling, hovering, and tracking—any physical manipulation or hardware degradation introduced by a cyber-physical attack can be isolated as a deviation from these established operational centroids.

To fulfill the requirements of the **Cyber Hypothesis** and complete the **Dual-Sensor architecture**, the physical kinematics must be contextualized within a network framework. For this, the research incorporates the **Tokyo Drone Communication and Security Dataset** [16], which documents comprehensive **drone-to-drone (D2D)** and **drone-to-base station (D2BS)** interactions within a technologically dense metropolitan area from **2018 to 2024**. This dataset exposes the intricate vulnerabilities of UAV communication topologies by recording high-resolution network features, including packet loss rates, jitter, cryptographic encryption states, signal-to-noise ratios, and onboard resource utilization (CPU and memory allocation). Furthermore, it provides labeled instances of severe network anomalies, including **Distributed Denial of Service (DDoS)** attacks and **malware infections**. By synthesizing the high-dimensional network traffic profiles of the **Tokyo dataset** with the detailed aerodynamic kinematics of the **SurveilDrone**, we establish a comprehensive, **dual-domain baseline**. This provides the foundation required for the subsequent feature engineering pipeline, allowing the raw metrics to be transformed into the dense, **195-dimensional temporal matrix** required for **zero-day threat isolation**.

## 4. Methods

Because zero-day cyber-physical threats inherently lack historical labels, supervised learning paradigms are unviable for this architecture. Consequently, this system was made utilizing **unsupervised machine learning methodologies**. To establish a comparative framework, the evaluation was structured across **six distinct experiments**, targeting a baseline contamination **rate of 5%**. The methodology is divided into three sequential phases: **baseline algorithmic evaluation**, **ensemble tribunal construction**, and **optimization**.

To satisfy the requirement for diversity, **four distinct algorithmic architectures** were evaluated against the **195-dimensional temporal matrix**:

- **Experiment 1 (Isolation Forest):** This tree-based algorithm isolates anomalies through recursive, random orthogonal partitioning. Because it relies on the principle that anomalies require fewer splits to be isolated than normal points, it is natively resistant to the computational overhead and distance-calculation flaws typical of high-dimensional spaces. [4]

- **Experiment 2 (One-Class SVM):** A boundary-driven methodology [9] utilizing a **Radial Basis Function (RBF) kernel**. This algorithm mapped the dual-sensor telemetry into a higher-dimensional, non-linear feature space, constructing a strict hypersphere that separated the dense normal operating telemetry from the origin.

- **Experiment 3 (LOF paired with PCA):** To test localized density deviations, we implemented the **Local Outlier Factor** [3]. However, because density-based metrics suffer severely from dimensional sparsity, this experiment required a precursor **PCA** step to compress the **195-dimensional matrix** into its most significant vectors before calculating local reachability distances.

- **Experiment 6 (K-Means Hybrid):** A geometric approach that clustered the telemetry to define normal aerodynamic and network **"operating states" (centroids)**. Anomalies were classified by calculating the maximum **Euclidean distances** of points from their respective operational centroids.

While the individual baseline algorithms successfully identified severe deviations, they each generated approximately **450 anomalies** across the flight log. In a live operational setting, this volume induces severe alert fatigue. To systematically suppress systemic noise and overcome the blind spots of individual algorithms, we developed two relational **meta-ensembles** via inner-joins on the temporal index:

- **Experiment 4 (The Strict Tribunal):** This architecture functioned as an intersection ensemble. It required unanimous geometric consensus across the baseline models to classify a timestamp as anomalous. This approach prioritized absolute precision and the total elimination of false positives, though it inherently risked ignoring sophisticated, multi-vector zero-days due to algorithm-specific blind spots.

- **Experiment 5 (The Meta-Ensemble):** To balance precision with recall, a **Majority-Vote Tribunal** was implemented, requiring a two-thirds agreement threshold across the models. This relaxed ensemble ensured that highly complex attacks—which might successfully evade the rigid boundaries of a **One-Class SVM** but trigger the density sensors of an **LOF**—were accurately escalated for human review.

By comparatively evaluating the unsupervised decision boundaries and the resistance to dimensional collapse, the **Isolation Forest (Experiment 1)** was selected as the **primary model**. To tune this baseline, we introduced a custom evaluation metric: the `decision_boundary_gap` (the numerical distance between the mean anomaly score and the mean normal score).

To maximize this boundary, we utilized **Optuna**, a next-generation **Bayesian hyperparameter optimization** framework [1]. Over **15 unsupervised trials**, Optuna navigated the hyperparameter search space, ultimately seeking the configuration that pushed true anomalies as far into the margins as possible. The optimization revealed that aggressively sub-sampling the dataset (restricting the trees to only 50% of the features and samples per split) forced the generation of highly diverse, uncorrelated boundary rules. This prevented the model from overfitting the dense normal cluster and successfully maximized the isolation of zero-day threats.

### 4.1 Hypotheses stating

The architectural design of this unsupervised detection framework is driven by **three hypotheses**, which collectively address the blind spots of traditional, siloed IDS. The first is the **Kinematic Hypothesis**. This postulates that physical hardware degradation or unauthorized flight control manipulations will produce distinct, quantifiable signatures in a UAV's physical telemetry prior to catastrophic systemic failure. During normal operations (e.g., patrolling or tracking), parameters such as spatial velocity, tri-axial acceleration, and hover duration operate within strict aerodynamic and power-drain bounds. When an adversary executes a zero-day cyber-physical attack—such as GPS spoofing or forcing the drone into an unauthorized, latent holding pattern—these physical metrics will inevitably deviate from their established mechanical baselines. Therefore, by establishing an unsupervised geometric boundary around normal kinematic behavior, physical hijacking can be mathematically detected as an anomalous spatial trajectory, even if the initiating cyber-vector remains entirely unclassified.

The second is the **Cyber Hypothesis**, addresses the inherent limitations of signature-based network security. This hypothesis asserts that latent network threats—including zero-day malware infections, **command-and-control (C&C)** hijacking, and Distributed Denial of Service (DDoS) floods—manifest as statistically significant outliers in network metadata, regardless of whether the malicious payload is known, documented, or encrypted. By analyzing high-resolution network traffic profiles such as packet loss rates, transmission jitter, protocol shifts, and sudden drops in cryptographic encryption states, the system can isolate hostile network behavior. This approach completely eliminates the dependency on outdated malware hash databases, relying instead on the principle that adversarial network intrusion inherently disrupts the steady-state communication frequency, **signal-to-noise ratio (SNR)**, and onboard CPU utilization of the drone's hardware.

The third and most critical for overcoming operational alert fatigue is the **Temporal State Hypothesis**. Traditional point-in-time anomaly detection evaluates each telemetry log in isolation, a method that inherently flags benign, split-second sensor noise as critical threats. This hypothesis posits that true cyber-physical anomalies are sustained, escalating events rather than instantaneous glitches; consequently, evaluating a UAV’s operational integrity requires the embedding of historical memory. By expanding the feature space using **rolling statistical windows** (e.g., 5-hour moving averages and standard deviations) alongside sequential lag variables, the model evaluates the UAV’s current state relative to its short-term historical trajectory. This temporal fusion ensures that isolated transmission drops are safely dismissed as environmental noise, while sustained, multi-vector deviations are accurately classified as zero-day attacks, thereby drastically reducing false-positive rates in live monitoring environments.

## References

[1] Akiba, Sano, Yanase, Ohta, Koyama. (2019). Optuna: A Next-generation Hyperparameter Optimization Framework. KDD 2019 Applied Data Science track. 10.48550/arXiv.1907.10902.

[2] Ankarboina, Harinath & Kumari, Jasmini & Singh, Amit. (2025). A Comprehensive Review on the Detection Capabilities of IDS Using Deep Learning Techniques. 10.1007/978-3-031-90723-4_1.

[3] Breunig, Kriegel, Raymond, Sander. (2000). LOF: identifying density-based local outliers. ACM SIGMOD Record, Volume 29, Issue 2. 10.1145/335191.335388.

[4] Fei Tony Liu, Kai Ming Ting, Zhi-Hua Zhou. (2008). Isolation Forest. 2008 Eighth IEEE International Conference on Data Mining. 10.1109/ICDM.2008.17.

[5] Hassan, Noureddine, Datta, Bates. (2020). OmegaLog: High-Fidelity Attack Investigation via Transparent Multi-layer Log Analysis. Network and Distributed Systems Security (NDSS) Symposium 2020. 10.14722/ndss.2020.24270.

[6] Khan, Liew, Yairi, McWilliam. (2019). Unsupervised anomaly detection in unmanned aerial vehicles. Periodicals, Applied Soft Computing. 10.1016/j.asoc.2019.105650.

[7] Kumar, Kuldeep, Tanwar, Namrta. (2025). The Adaptive Ensemble Learning-Based Intrusion Detection System for Enhanced Cybersecurity in Networked Environments. 10.21203/rs.3.rs-7385506/v1.

[8] Papathanasiou, Zacharakis, Liaperdos, Kotsilieris, Livieris, Ioannou. (2026). Secure Communication Protocols and AI-Based Anomaly Detection in UAV-GCS. MDPI, Applied Sciences. 10.3390/app16073339.

[9] Scholkopf, Plattz, Taylory, Smolax, Williamsonx. (2000). Estimating the Support of a High-Dimensional Distribution. Department of Engineering, Australian National University, Canberra 0200, Australia. MSR-TR-99-87.

[10] Sezgin, Anıl & Keskin, Rasim & Boyacı, Aytuğ. (2025). Anomaly Detection in Unmanned Aerial Vehicle Telemetry Using Automated Machine Learning. Fırat Üniversitesi Mühendislik Bilimleri Dergisi. 37. 699-709. 10.35234/fumbd.1668498.

[11] Yang, Lei & Li, ShaoBo & Li, Chuanjiang & Zhu, CaiChao & Zhang, AnSi & Liang, GuoQiang. (2023). Data-driven unsupervised anomaly detection and recovery of unmanned aerial vehicle flight data based on spatiotemporal correlation. Science China Technological Sciences. 66. 1304-1316. 10.1007/s11431-022-2312-8.

[12] https://datos.gob.es/sites/default/files/documentacion/files/Guia_DVC_en.pdf.

[13] https://docs.evidentlyai.com/introduction.

[14] https://indico.kit.edu/event/3845/contributions/14701/attachments/6987/11043/1.6_MLflow_and_its_usage.pdf.

[15] https://www.kaggle.com/datasets/datasetengineer/surveildrone-net23.

[16] https://www.kaggle.com/datasets/datasetengineer/tokyo-drone-communication-and-security-dataset/data.